# Imports

In [1]:
import re
import cda2
import math

import time
from datetime import datetime, timedelta

import pandas as pd
import pyarrow
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf

from pyspark.sql.window import Window
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import explode, map_keys, col

In [2]:
api = cda2.Api()

In [3]:
# Set configuration parameters to better optimize queries.

config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

In [4]:
# Start Spark and specify number of cpus to use.

api.start_spark(n_executors=100, config=config)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bd832029-7be5-427a-850d-30d6c336f1ef;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

In [5]:
year0 = "2026"
year1 = str(int(year0) + 1)

In [6]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

# retrieve tracons

In [7]:
df_airspaces = (
    api.dataframe("StaticEramAirspace",
                  **dates,
#                  partition_filters=api.custom_partitions(['ARTCC_SECTOR']), // additional: {FULLNAME -> SECT1} 
#                  partition_filters=api.custom_partitions(['SECTOR']), // additional: {ACTIVE -> false} or {ACTIVE -> true}
#                  partition_filters=api.custom_partitions(['TRACON']), // additional: {}
#                  partition_filters=api.custom_partitions(['SAA']), // additional: {ACTIVE -> ALWAYS_OFF, TYPE -> LOCAL_ASSIGNED}
#                  partition_filters=api.custom_partitions(['FAV']), # additional: {TYPE -> ENROUTE} ; {TYPE -> APPROACH, ARTSID -> MCI} ; 
#                  partition_filters=api.custom_partitions(['ZDC']),
                  partition_filters=api.custom_partitions(['TRACON']),
                  metadata=True)
    .select (
        "center",
#        "type",
        "identifier",
        "additional",
        F.explode('geometry.modules').alias('module'),
        "chart_date"
    )
    .withColumnRenamed('center', 'artcc')
    .withColumnRenamed('identifier', 'tracon')
    .withColumn("grouping", F.concat("artcc", F.lit("_"), "tracon", F.lit("_"), "module.identifier"))
    .orderBy("grouping")
)

Could not find data for the following date ranges: 
    (2026-03-19, 2027-01-01)
Multiple versions found: 3.1.79, 3.1.81
                                                                                

In [8]:
df_airspaces = (df_airspaces
    .drop("additional")
)

In [9]:
df_airspaces.count()

2848

In [10]:
df_airspaces.show()

+-----+------+--------------------+-------------+---------------+
|artcc|tracon|              module|   chart_date|       grouping|
+-----+------+--------------------+-------------+---------------+
|  ZAB|   ABQ|{00201-1, {[{34.8...|1764201600000|ZAB_ABQ_00201-1|
|  ZAB|   ABQ|{00201-1, {[{34.8...|1769040000000|ZAB_ABQ_00201-1|
|  ZAB|   ABQ|{00210-1, {[{34.8...|1764201600000|ZAB_ABQ_00210-1|
|  ZAB|   ABQ|{00210-1, {[{34.8...|1769040000000|ZAB_ABQ_00210-1|
|  ZAB|   ABQ|{00217-1, {[{34.8...|1764201600000|ZAB_ABQ_00217-1|
|  ZAB|   ABQ|{00217-1, {[{34.8...|1769040000000|ZAB_ABQ_00217-1|
|  ZAB|   ABQ|{00230-1, {[{35.1...|1764201600000|ZAB_ABQ_00230-1|
|  ZAB|   ABQ|{00230-1, {[{35.1...|1769040000000|ZAB_ABQ_00230-1|
|  ZAB|   ABQ|{00231-1, {[{35.4...|1764201600000|ZAB_ABQ_00231-1|
|  ZAB|   ABQ|{00231-1, {[{35.4...|1769040000000|ZAB_ABQ_00231-1|
|  ZAB|   ABQ|{00234-1, {[{35.1...|1764201600000|ZAB_ABQ_00234-1|
|  ZAB|   ABQ|{00234-1, {[{35.1...|1769040000000|ZAB_ABQ_00234-1|
|  ZAB|   

## keep only the newest update

In [11]:
#window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#window = Window.partitionBy("grouping").orderBy(col("metadata.effective_end_date").desc())
window = Window.partitionBy("grouping").orderBy(col("chart_date").desc())

df_airspaces_newest = (df_airspaces
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "chart_date")
    .orderBy("grouping")
)

In [12]:
df_airspaces_newest.count()

1427

In [13]:
df_airspaces_newest.show()

+-----+------+--------------------+---------------+
|artcc|tracon|              module|       grouping|
+-----+------+--------------------+---------------+
|  ZAB|   ABQ|{00201-1, {[{34.8...|ZAB_ABQ_00201-1|
|  ZAB|   ABQ|{00210-1, {[{34.8...|ZAB_ABQ_00210-1|
|  ZAB|   ABQ|{00217-1, {[{34.8...|ZAB_ABQ_00217-1|
|  ZAB|   ABQ|{00230-1, {[{35.1...|ZAB_ABQ_00230-1|
|  ZAB|   ABQ|{00231-1, {[{35.4...|ZAB_ABQ_00231-1|
|  ZAB|   ABQ|{00234-1, {[{35.1...|ZAB_ABQ_00234-1|
|  ZAB|   ABQ|{00236-1, {[{35.4...|ZAB_ABQ_00236-1|
|  ZAB|   ABQ|{00237-1, {[{35.3...|ZAB_ABQ_00237-1|
|  ZAB|   ABQ|{00240-1, {[{35.6...|ZAB_ABQ_00240-1|
|  ZAB|   ABQ|{00294-1, {[{34.8...|ZAB_ABQ_00294-1|
|  ZAB|   AMA|{00601-1, {[{34.7...|ZAB_AMA_00601-1|
|  ZAB|   AMA|{00601-2, {[{34.7...|ZAB_AMA_00601-2|
|  ZAB|   CVS|{00801-1, {[{34.6...|ZAB_CVS_00801-1|
|  ZAB|   ELP|{00101-1, {[{31.7...|ZAB_ELP_00101-1|
|  ZAB|   ELP|{00131-1, {[{32.2...|ZAB_ELP_00131-1|
|  ZAB|   ELP|{00132-1, {[{31.7...|ZAB_ELP_00132-1|
|  ZAB|   FH

In [14]:
module_schema = T.StructType([
    T.StructField("module", T.StructType([
        T.StructField("identifier", T.DoubleType(), True),
        T.StructField("floor", T.DoubleType(), True),
        T.StructField("ceiling", T.DoubleType(), True),
        T.StructField("polygon", T.StructType([
            T.StructField("boundary", T.ArrayType(
                T.StructType([
                    T.StructField("latitude", T.DoubleType(), True),
                    T.StructField("longitude", T.DoubleType(), True),
                    T.StructField("sequence_number", T.IntegerType(), True),
                ]), True), True),
        ]), True),
    ]), True),
])

In [15]:
none_string = "*"
separator_string = ":"

#@F.udf(T.StringType())
@F.udf(T.MapType(T.StringType(),T.StringType()))
def extract_boundary_from_module(module:module_schema) -> map:  
    result = {}

    result["shelf"] = f'{module.identifier}'
    result["floor"] = f'{module.floor:.0f}'
    result["ceiling"] = f'{module.ceiling:.0f}'

    boundary_string = ""

    module.polygon.boundary.sort(key=lambda x: x.sequence_number, reverse=False)

    for point in module.polygon.boundary:
        boundary_string += (none_string if point.latitude is None else f'{point.latitude:.6f}') + " "
        boundary_string += (none_string if point.longitude is None else f'{point.longitude:.6f}') + separator_string

    # drop the trailing separator
    if len(boundary_string) > 0:
        boundary_string = boundary_string[0:len(separator_string) * -1]

    result["boundary"] = boundary_string
    result["shelf"] = module.identifier

    return result

In [16]:
df_airspaces_dict = (
    df_airspaces_newest
    .withColumn("module_dict", extract_boundary_from_module("module"))
    .select(
        "artcc",
#        "type",
        "tracon",
        "module_dict",
    )
    .drop("module")
    .orderBy("artcc", "tracon")
)

In [17]:
#df_tracons_dict.show(1, truncate=False)

In [18]:
# expand the dictionary to columns

# https://mungingdata.com/pyspark/dict-map-to-multiple-columns/
# https://stackoverflow.com/questions/36869134/pyspark-converting-a-column-of-type-map-to-multiple-columns-in-a-dataframe

keys = ["shelf", "ceiling", "floor", "boundary"]
key_cols = list(map(lambda f: F.col("module_dict").getItem(f).alias(str(f)), keys))
final_cols = [
        "artcc",
#        "type",
        "tracon",
             ] + key_cols

In [19]:
df_airspaces_output = (
    df_airspaces_dict.select(final_cols)
    .withColumnRenamed('boundary', 'geometry')
)

In [20]:
#df_airspaces_output.show(10)

In [21]:
(
    df_airspaces_output
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/tracons", compression="None", mode="overwrite")
)